<a href="https://colab.research.google.com/github/LuizFellipiFreire25/Projeto-ECAA08/blob/main/etapa-02-grafos/12%20-%20Matrizes%20de%20Incidencia%20Adjacencia%20e%20Custos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aula 12 - Notebook: Matriz de Incidência e Balanço de Tráfego de AGVs
# Projeto: SCADA-Core / Visão-AGV
# Neste notebook implementamos a Matriz de Incidência Vértice-Aresta B e resolvemos o balanço matricial de tráfego B * Q = S na malha logística autônoma.

In [1]:
def formatar_tabela(dados):
    """Formata lista de dicionarios em tabela ASCII pura."""
    if not dados:
        return "Tabela Vazia"
    colunas = list(dados[0].keys())
    larguras = {c: len(str(c)) for c in colunas}
    for row in dados:
        for c in colunas:
            larguras[c] = max(larguras[c], len(str(row.get(c, ""))))
    header = " | ".join(f"{c:<{larguras[c]}}" for c in colunas)
    divisor = "-+-".join("-" * larguras[c] for c in colunas)
    linhas = [header, divisor]
    for row in dados:
        linhas.append(" | ".join(f"{str(row.get(c, '')):<{larguras[c]}}" for c in colunas))
    return "\n".join(linhas)

def formatar_matriz(matriz, rotulos_linhas, rotulos_cols):
    """Formata matriz 2D em tabela ASCII pura."""
    larguras = [max(len(str(r)), 6) for r in rotulos_cols]
    larg_linha = max(len(str(r)) for r in rotulos_linhas)
    header = f"{' ' * larg_linha} | " + " | ".join(f"{c:>{larguras[j]}}" for j, c in enumerate(rotulos_cols))
    divisor = f"{'-' * larg_linha}-+-" + "-+-".join("-" * larguras[j] for j in range(len(rotulos_cols)))
    linhas = [header, divisor]
    for i, r_nome in enumerate(rotulos_linhas):
        vals = []
        for j in range(len(rotulos_cols)):
            v = matriz[i][j]
            v_str = "∞" if v == float('inf') else str(v)
            vals.append(f"{v_str:>{larguras[j]}}")
        linhas.append(f"{r_nome:<{larg_linha}} | " + " | ".join(vals))
    return "\n".join(linhas)

class GrafoMalhaAGV:
    def __init__(self, vertices):
        self.vertices = vertices
        self.v_to_idx = {v: i for i, v in enumerate(vertices)}
        self.idx_to_v = {i: v for i, v in enumerate(vertices)}
        self.n = len(vertices)
        self.adj_binaria = [[0] * self.n for _ in range(self.n)]
        self.adj_pesos = [[float('inf')] * self.n for _ in range(self.n)]
        for i in range(self.n): self.adj_pesos[i][i] = 0.0
        self.arestas_detalhes = []

    def adicionar_rota(self, origem, destino, comprimento_m, tag_rfid, largura_corredor_m=2.5):
        u = self.v_to_idx[origem]
        v = self.v_to_idx[destino]
        self.adj_binaria[u][v] = 1
        self.adj_pesos[u][v] = comprimento_m
        self.arestas_detalhes.append({
            "Origem": origem,
            "Destino": destino,
            "Comprimento (m)": comprimento_m,
            "Tag RFID/LiDAR": tag_rfid,
            "Largura (m)": largura_corredor_m
        })

def criar_malha_agv_padrao():
    estacoes = ["ST-01", "DOC-101", "ALM-201", "AMO-301", "R-101", "DEP-401"]
    g = GrafoMalhaAGV(estacoes)
    g.adicionar_rota("ST-01", "DOC-101", 10.0, "TAG-01", 2.0)
    g.adicionar_rota("ST-01", "ALM-201", 12.0, "TAG-02", 2.0)
    g.adicionar_rota("DOC-101", "ALM-201", 15.0, "TAG-03", 2.0)
    g.adicionar_rota("ALM-201", "AMO-301", 20.0, "TAG-04", 2.5)
    g.adicionar_rota("AMO-301", "R-101", 18.0, "TAG-05", 2.5)
    g.adicionar_rota("AMO-301", "DEP-401", 14.0, "TAG-06", 2.0)
    g.adicionar_rota("R-101", "DEP-401", 25.0, "TAG-07", 3.0)
    g.adicionar_rota("DEP-401", "ST-01", 30.0, "TAG-08", 3.0)
    return g

malha = criar_malha_agv_padrao()

class CalculadorIncidencia:
    @staticmethod
    def construir_matriz_incidencia(vertices, arestas):
        n = len(vertices)
        m = len(arestas)
        v_idx = {v: i for i, v in enumerate(vertices)}
        B = [[0] * m for _ in range(n)]
        nomes_e = []
        for j, a in enumerate(arestas):
            u = v_idx[a["Origem"]]
            v = v_idx[a["Destino"]]
            B[u][j] = -1
            B[v][j] = 1
            nomes_e.append(f"e{j+1}:{a['Origem']}->{a['Destino']}")
        return B, nomes_e

B_mat, nomes_arestas = CalculadorIncidencia.construir_matriz_incidencia(malha.vertices, malha.arestas_detalhes)
print("=== MATRIZ DE INCIDÊNCIA B (MALHA LOGÍSTICA AGV) ===")
print(formatar_matriz(B_mat, malha.vertices, [f"e{j+1}" for j in range(len(malha.arestas_detalhes))]))

# Verificação de propriedade formal: soma por coluna deve ser nula
somas_col = [sum(B_mat[i][j] for i in range(len(malha.vertices))) for j in range(len(malha.arestas_detalhes))]
assert all(s == 0 for s in somas_col)

# Fluxo de Tráfego de AGVs Q (veículos/h em cada rota e1..e8)
# e1: 4.0, e2: 6.0, e3: 4.0, e4: 10.0, e5: 6.0, e6: 4.0, e7: 6.0, e8: 10.0
Q_vec = [4.0, 6.0, 4.0, 10.0, 6.0, 4.0, 6.0, 10.0]

S_res = []
for i, v_nome in enumerate(malha.vertices):
    balanco = sum(B_mat[i][j] * Q_vec[j] for j in range(len(Q_vec)))
    S_res.append({"Estação": v_nome, "Balanço Líquido (AGVs/h)": f"{balanco:.1f}"})

print("\n--- Balanço de Tráfego em Regime Permanente (S = B * Q) ---")
print(formatar_tabela(S_res))

=== MATRIZ DE INCIDÊNCIA B (MALHA LOGÍSTICA AGV) ===
        |     e1 |     e2 |     e3 |     e4 |     e5 |     e6 |     e7 |     e8
--------+--------+--------+--------+--------+--------+--------+--------+-------
ST-01   |     -1 |     -1 |      0 |      0 |      0 |      0 |      0 |      1
DOC-101 |      1 |      0 |     -1 |      0 |      0 |      0 |      0 |      0
ALM-201 |      0 |      1 |      1 |     -1 |      0 |      0 |      0 |      0
AMO-301 |      0 |      0 |      0 |      1 |     -1 |     -1 |      0 |      0
R-101   |      0 |      0 |      0 |      0 |      1 |      0 |     -1 |      0
DEP-401 |      0 |      0 |      0 |      0 |      0 |      1 |      1 |     -1

--- Balanço de Tráfego em Regime Permanente (S = B * Q) ---
Estação | Balanço Líquido (AGVs/h)
--------+-------------------------
ST-01   | 0.0                     
DOC-101 | 0.0                     
ALM-201 | 0.0                     
AMO-301 | 0.0                     
R-101   | 0.0                     
D